# TALLER 3 — FLUJO ANALÍTICO REPRODUCIBLE

### Requisito:
La calificación del Taller Práctico está condicionada a la entrega previa del vídeo semanal.

### Reto
Diseñar un flujo analítico reproducible.

### Objetivo y alcance del reto:
Transformar un proceso manual de consolidación de datos en un flujo reproducible. Se aplica agrupamiento, agregación, combinación de datasets, automatización de archivos, gestión de entornos, organización de proyectos y control de versiones.

### Modalidad del entregable:
Escrito — permite documentar decisiones técnicas, justificar criterios de reproducibilidad, presentar la estructura del proyecto y explicar validaciones de forma ordenada.

### 1.1 Configuración del entorno

In [19]:
#? Importamos las dependencias mínimas para el flujo completo.
#? sys: verificar versión de Python; pathlib: rutas portables; datetime: sellos de tiempo en exports.
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('--- ENTORNO ---')
print(f'Python: {sys.version.split()[0]}')
print(f'Pandas: {pd.__version__}')
print(f'Numpy:  {np.__version__}')

--- ENTORNO ---
Python: 3.12.13
Pandas: 2.2.3
Numpy:  2.0.2


### 1.2 Arquitectura de carpetas

In [20]:
#? Rutas relativas con pathlib: el proyecto corre en cualquier máquina sin editar paths.
#? Separamos raw (intocable) de processed (generado) para trazabilidad.
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

for directorio in [RAW_DIR, PROCESSED_DIR]:
    directorio.mkdir(parents=True, exist_ok=True)

print('--- ESTRUCTURA ---')
print(f'Raw:       {RAW_DIR}')
print(f'Processed: {PROCESSED_DIR}')

--- ESTRUCTURA ---
Raw:       /content/data/raw
Processed: /content/data/processed


### 1.3 Funciones reutilizables

In [21]:
#? Centralizamos lógica repetitiva en funciones para no duplicar código entre secciones.
#? Cada función encapsula una validación que se aplica a múltiples DataFrames.

def leer_archivo_seguro(ruta, tipo='csv', **kwargs):
    """Lee CSV, Excel o JSON de forma robusta."""
    if tipo == 'csv':
        return pd.read_csv(ruta, **kwargs)
    elif tipo == 'excel':
        return pd.read_excel(ruta, **kwargs)
    elif tipo == 'json':
        return pd.read_json(ruta, **kwargs)
    else:
        raise ValueError(f'Tipo no soportado: {tipo}')


def validar_columnas(df, columnas_requeridas, nombre_df='DataFrame'):
    """Falla temprano si faltan columnas esperadas."""
    faltantes = [c for c in columnas_requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f'{nombre_df} sin columnas: {faltantes}')
    print(f'   ✓ {nombre_df}: columnas validadas')
    return True


def detectar_duplicados(df, subset, nombre_df='DataFrame'):
    """Reporta duplicados sobre un subset de columnas."""
    dupes = df[df.duplicated(subset=subset, keep=False)]
    n = len(dupes)
    print(f'   {nombre_df}: {n} registros duplicados en {subset}')
    return dupes


def contar_nulos(df, columna, nombre_df='DataFrame'):
    """Cuenta nulos en una columna y reporta porcentaje."""
    n = df[columna].isna().sum()
    pct = n / len(df) * 100
    print(f'   {nombre_df} → {columna}: {n} nulos ({pct:.1f}%)')
    return n

print('--- FUNCIONES CARGADAS ---')
print('leer_archivo_seguro, validar_columnas, detectar_duplicados, contar_nulos')

--- FUNCIONES CARGADAS ---
leer_archivo_seguro, validar_columnas, detectar_duplicados, contar_nulos


In [22]:
#? Copiamos los archivos fuente al directorio raw para garantizar trazabilidad.
#  Esto asegura que los datos originales estén dentro del proyecto.
import shutil

# Archivos fuente requeridos para el análisis
archivos_fuente = {
    'entregas_campus_matriz.csv': BASE_DIR / 'entregas_campus_matriz.csv',
    'entregas_campus_extension.csv': BASE_DIR / 'entregas_campus_extension.csv',
    'estudiantes_master.xlsx': BASE_DIR / 'estudiantes_master.xlsx',
}

print('--- COPIADO DE ARCHIVOS A RAW ---')
for nombre, origen in archivos_fuente.items():
    destino = RAW_DIR / nombre
    if origen.exists():
        shutil.copy2(origen, destino)
        print(f'  ✓ {nombre} copiado a data/raw/')
    else:
        print(f'  ✗ {nombre} NO encontrado en {origen}')

print('--- FIN COPIADO ---')

--- COPIADO DE ARCHIVOS A RAW ---
  ✓ entregas_campus_matriz.csv copiado a data/raw/
  ✓ entregas_campus_extension.csv copiado a data/raw/
  ✓ estudiantes_master.xlsx copiado a data/raw/
--- FIN COPIADO ---


### 2.1 Lectura estructurada de datos

In [23]:
#? Forzamos dtype str en IDs para evitar que pandas los interprete como numéricos y pierda ceros.
#? Fallback CSV → si el Excel está corrupto el flujo no se detiene.

print('--- CARGA DE DATOS ---')

# Catálogo maestro (Excel)
ruta_catalogo = RAW_DIR / 'estudiantes_master.xlsx'
try:
    df_catalogo = leer_archivo_seguro(
        ruta_catalogo, tipo='excel', engine='openpyxl',
        dtype={'id_estudiante': str}
    )
except Exception:
    df_catalogo = leer_archivo_seguro(
        ruta_catalogo, tipo='csv', sep=',',
        dtype={'id_estudiante': str}
    )

columnas_catalogo = ['id_estudiante', 'nombre_completo', 'campus_origen']
validar_columnas(df_catalogo, columnas_catalogo, 'Catálogo')

#? Eliminamos duplicados en la llave primaria antes del merge.
df_catalogo_limpio = df_catalogo.drop_duplicates(subset=['id_estudiante'])
print(f'   Catálogo: {len(df_catalogo)} → {len(df_catalogo_limpio)} (sin duplicados)')

# Entregas campus Matriz (CSV)
df_matriz = leer_archivo_seguro(
    RAW_DIR / 'entregas_campus_matriz.csv',
    tipo='csv', sep=',',
    dtype={'id_entrega': str, 'id_estudiante': str}
)

# Entregas campus Extensión (CSV)
df_extension = leer_archivo_seguro(
    RAW_DIR / 'entregas_campus_extension.csv',
    tipo='csv', sep=',',
    dtype={'id_entrega': str, 'id_estudiante': str}
)

columnas_entregas = ['id_entrega', 'id_estudiante', 'fecha_subida',
                     'materia', 'tipo_proyecto', 'puntaje_obtenido', 'estado_entrega']
validar_columnas(df_matriz, columnas_entregas, 'Matriz')
validar_columnas(df_extension, columnas_entregas, 'Extensión')

print(f'\n   Matriz:    {df_matriz.shape}')
print(f'   Extensión: {df_extension.shape}')
print(f'   Catálogo:  {df_catalogo_limpio.shape}')

--- CARGA DE DATOS ---
   ✓ Catálogo: columnas validadas
   Catálogo: 60 → 60 (sin duplicados)
   ✓ Matriz: columnas validadas
   ✓ Extensión: columnas validadas

   Matriz:    (150, 7)
   Extensión: (150, 7)
   Catálogo:  (60, 7)


### 2.2 Integración estructural (concat)

In [24]:
#? Concat: unimos DataFrames con esquema idéntico (mismas columnas, distinto origen).
#? Agregamos columna 'campus' ANTES de concatenar para no perder el origen de cada fila.

df_matriz['campus'] = 'matriz'
df_extension['campus'] = 'extension'

df_consolidado = pd.concat(
    [df_matriz, df_extension],
    ignore_index=True
)

#? Verificamos que no haya IDs de entrega repetidos entre campus.
detectar_duplicados(df_consolidado, subset=['id_entrega'], nombre_df='Consolidado')

print(f'\n--- CONCAT ---')
print(f'Matriz ({len(df_matriz)}) + Extensión ({len(df_extension)}) = {len(df_consolidado)} registros')

   Consolidado: 0 registros duplicados en ['id_entrega']

--- CONCAT ---
Matriz (150) + Extensión (150) = 300 registros


### 2.3 Integración relacional (merge)

In [25]:
#? Merge left: conservamos TODAS las entregas aunque el estudiante no esté en el catálogo.
#? validate='many_to_one': falla si el catálogo tiene claves duplicadas (detecta error upstream).
#? indicator=True: nos permite auditar qué registros no encontraron match.

df_integrado = pd.merge(
    df_consolidado,
    df_catalogo_limpio,
    on='id_estudiante',
    how='left',
    validate='many_to_one',
    indicator=True
)

print('--- MERGE ---')
print(df_integrado['_merge'].value_counts().to_string())

#? Huérfanos = entregas de estudiantes no matriculados. Los exportamos para auditoría.
df_huerfanos = df_integrado[df_integrado['_merge'] == 'left_only']
print(f'\nHuérfanos: {len(df_huerfanos)} ({len(df_huerfanos)/len(df_integrado)*100:.1f}%)')

if len(df_huerfanos) > 0:
    archivo_huerfanos = PROCESSED_DIR / f'huerfanos_{datetime.now().strftime("%Y%m%d")}.csv'
    df_huerfanos.to_csv(archivo_huerfanos, index=False)
    print(f'   Exportado → {archivo_huerfanos.name}')

df_integrado = df_integrado.drop(columns=['_merge'])

--- MERGE ---
_merge
both          295
left_only       5
right_only      0

Huérfanos: 5 (1.7%)
   Exportado → huerfanos_20260819.csv


### 2.4 Validaciones adicionales y limpieza

In [26]:
#? Identificamos nulos críticos en puntaje: pueden indicar entregas no calificadas.
#? También buscamos inconsistencias lógicas entre estado y puntaje.

print('--- VALIDACIONES ---')
nulos_puntaje = contar_nulos(df_integrado, 'puntaje_obtenido', 'Integrado')

# Inconsistencias: Pendiente con puntaje o Revisión sin puntaje
pendiente_con_nota = df_integrado[
    (df_integrado['estado_entrega'] == 'Pendiente') &
    (df_integrado['puntaje_obtenido'].notna())
]
revision_sin_nota = df_integrado[
    (df_integrado['estado_entrega'] == 'Revisión') &
    (df_integrado['puntaje_obtenido'].isna())
]

print(f'   Pendiente CON puntaje (inconsistente): {len(pendiente_con_nota)}')
print(f'   Revisión SIN puntaje (inconsistente):  {len(revision_sin_nota)}')

--- VALIDACIONES ---
   Integrado → puntaje_obtenido: 9 nulos (3.0%)
   Pendiente CON puntaje (inconsistente): 0
   Revisión SIN puntaje (inconsistente):  0


### 2.5 Análisis exploratorio y agregaciones

In [27]:
#? Agregamos por múltiples ejes para entender la distribución de los datos.
#? Esto responde preguntas de negocio: ¿qué campus rinde más? ¿qué materia tiene más varianza?

print('--- DESCRIPTIVOS ---')
print(df_integrado['puntaje_obtenido'].describe().to_string())

print('\n--- PROMEDIO POR CAMPUS ---')
print(df_integrado.groupby('campus')['puntaje_obtenido'].mean().to_string())

print('\n--- ESTADO DE ENTREGAS ---')
print(df_integrado['estado_entrega'].value_counts().to_string())

print('\n--- TOP 5 ESTUDIANTES (promedio) ---')
top5 = (df_integrado.groupby('id_estudiante')['puntaje_obtenido']
        .mean().nlargest(5))
print(top5.to_string())

print('\n--- BOTTOM 5 ESTUDIANTES (promedio) ---')
bottom5 = (df_integrado.groupby('id_estudiante')['puntaje_obtenido']
           .mean().nsmallest(5))
print(bottom5.to_string())

print('\n--- PROMEDIO POR MATERIA ---')
print(df_integrado.groupby('materia')['puntaje_obtenido'].mean().sort_values(ascending=False).to_string())

print('\n--- PROMEDIO POR TIPO DE PROYECTO ---')
print(df_integrado.groupby('tipo_proyecto')['puntaje_obtenido'].mean().sort_values(ascending=False).to_string())

--- DESCRIPTIVOS ---
count    291.000000
mean      90.347079
std        6.627905
min       60.000000
25%       87.000000
50%       91.000000
75%       95.000000
max      100.000000

--- PROMEDIO POR CAMPUS ---
campus
extension    90.697279
matriz       89.989583

--- ESTADO DE ENTREGAS ---
estado_entrega
Aprobado     276
Revisión      15
Pendiente      9

--- TOP 5 ESTUDIANTES (promedio) ---
id_estudiante
E999    100.000000
E026     99.285714
E058     99.000000
E039     98.500000
E043     98.333333

--- BOTTOM 5 ESTUDIANTES (promedio) ---
id_estudiante
E888    60.000000
E012    70.333333
E030    76.714286
E020    81.800000
E042    83.000000

--- PROMEDIO POR MATERIA ---
materia
Desarrollo de Software        90.752427
DataOps                       90.532609
Sistemas de Bases de Datos    89.734375

--- PROMEDIO POR TIPO DE PROYECTO ---
tipo_proyecto
Testing          91.513514
Prototipo        91.483871
Frontend         91.000000
Pipeline         90.857143
Deploy           90.260000
Scrip

### 2.6 Resumen analítico por estudiante

In [28]:
#? Agrupamos por estudiante+campus para generar un perfil resumido por persona.
#? La desviación estándar revela consistencia: un alumno con std alta es irregular.

resumen_final = (df_integrado
    .groupby(['id_estudiante', 'campus'])
    .agg(
        promedio=('puntaje_obtenido', 'mean'),
        entregas=('id_entrega', 'count'),
        desviacion=('puntaje_obtenido', 'std')
    )
    .reset_index()
)

print('--- RESUMEN POR ESTUDIANTE ---')
print(f'Dimensiones: {resumen_final.shape}')
print(resumen_final.head(10).to_string(index=False))

--- RESUMEN POR ESTUDIANTE ---
Dimensiones: (64, 5)
id_estudiante    campus  promedio  entregas  desviacion
         E001    matriz 93.785714         7    4.376615
         E002    matriz 83.857143         7    7.358183
         E003    matriz 91.600000         7    1.557241
         E004 extension 89.857143         8    1.772811
         E005 extension 94.250000         8    1.982062
         E006    matriz 96.857143         7    3.023716
         E007    matriz 85.500000         7    4.330127
         E008    matriz 94.142857         7    3.023716
         E009 extension 87.562500         8    1.678381
         E010 extension 97.875000         8    1.246423


### 2.7 Indicadores de calidad del proceso

In [29]:
#? Los indicadores auditan el pipeline: si alguno cambia entre ejecuciones, algo se rompió.
#? Exportarlos permite comparar corridas y detectar regresiones en los datos fuente.

indicadores = {
    'archivos_procesados': 3,
    'registros_catalogo': len(df_catalogo),
    'registros_matriz': len(df_matriz),
    'registros_extension': len(df_extension),
    'registros_consolidados': len(df_consolidado),
    'registros_integrados': len(df_integrado),
    'duplicados_catalogo_eliminados': len(df_catalogo) - len(df_catalogo_limpio),
    'claves_sin_correspondencia': len(df_huerfanos),
    'nulos_puntaje_obtenido': nulos_puntaje,
    'dimension_salida_filas': df_integrado.shape[0],
    'dimension_salida_columnas': df_integrado.shape[1],
    'archivos_generados': 0  # se actualiza abajo
}

print('--- INDICADORES DE CALIDAD ---')
for k, v in indicadores.items():
    print(f'   {k:.<40} {v}')

df_indicadores = pd.DataFrame([indicadores])
nombre_ind = PROCESSED_DIR / f'indicadores_calidad_{datetime.now().strftime("%Y%m%d")}.csv'
df_indicadores.to_csv(nombre_ind, index=False)
print(f'\n   Exportado → {nombre_ind.name}')

--- INDICADORES DE CALIDAD ---
   archivos_procesados..................... 3
   registros_catalogo...................... 60
   registros_matriz........................ 150
   registros_extension..................... 150
   registros_consolidados.................. 300
   registros_integrados.................... 300
   duplicados_catalogo_eliminados.......... 0
   claves_sin_correspondencia.............. 5
   nulos_puntaje_obtenido.................. 9
   dimension_salida_filas.................. 300
   dimension_salida_columnas............... 14
   archivos_generados...................... 0

   Exportado → indicadores_calidad_20260819.csv


### 2.8 Exportación de datos finales

In [30]:
#? Exportamos en 3 formatos para cubrir distintos consumidores downstream.
#? CSV: universal. JSON: APIs y NoSQL. Excel: usuarios no técnicos.

stamp = datetime.now().strftime('%Y%m%d')
archivos_exportados = []

# Dataset integrado
for fmt, method in [('csv', 'to_csv'), ('json', 'to_json'), ('xlsx', 'to_excel')]:
    nombre = PROCESSED_DIR / f'dataset_integrado_{stamp}.{fmt}'
    if fmt == 'json':
        df_integrado.to_json(nombre, orient='records', force_ascii=False, indent=2)
    elif fmt == 'xlsx':
        df_integrado.to_excel(nombre, index=False, engine='openpyxl')
    else:
        df_integrado.to_csv(nombre, index=False)
    archivos_exportados.append(nombre.name)

# Resumen por estudiante
for fmt, method in [('csv', 'to_csv'), ('json', 'to_json'), ('xlsx', 'to_excel')]:
    nombre = PROCESSED_DIR / f'resumen_estudiantes_{stamp}.{fmt}'
    if fmt == 'json':
        resumen_final.to_json(nombre, orient='records', force_ascii=False, indent=2)
    elif fmt == 'xlsx':
        resumen_final.to_excel(nombre, index=False, engine='openpyxl')
    else:
        resumen_final.to_csv(nombre, index=False)
    archivos_exportados.append(nombre.name)

print('--- EXPORTACIÓN ---')
for a in archivos_exportados:
    print(f'   ✓ {a}')
print(f'\n   Total archivos generados: {len(archivos_exportados) + 2}')  # +2: huérfanos + indicadores

--- EXPORTACIÓN ---
   ✓ dataset_integrado_20260819.csv
   ✓ dataset_integrado_20260819.json
   ✓ dataset_integrado_20260819.xlsx
   ✓ resumen_estudiantes_20260819.csv
   ✓ resumen_estudiantes_20260819.json
   ✓ resumen_estudiantes_20260819.xlsx

   Total archivos generados: 8


### 2.9 Estrategia de reproducibilidad

In [31]:
#? Documentamos las condiciones para que otro analista reproduzca el resultado sin ayuda verbal.

print("""--- ESTRATEGIA DE REPRODUCIBILIDAD ---

1. Entorno virtual:
   python -m venv .venv
   source .venv/bin/activate

2. Dependencias (requirements.txt):
   pandas==2.2.2
   numpy==2.2.2
   openpyxl==3.1.5

3. Control de versiones (Git):
   - .gitignore: data/raw/*, *.pyc, .venv/, .ipynb_checkpoints/
   - Commits frecuentes con mensajes descriptivos

4. Ejecución:
   - Rutas relativas (pathlib) → portable
   - Para Colab: montar Drive y copiar archivos a data/raw/

5. Determinismo:
   - Sin operaciones aleatorias
   - Timestamps en nombres de archivos para trazabilidad
""")

--- ESTRATEGIA DE REPRODUCIBILIDAD ---

1. Entorno virtual:
   python -m venv .venv
   source .venv/bin/activate

2. Dependencias (requirements.txt):
   pandas==2.2.2
   numpy==2.2.2
   openpyxl==3.1.5

3. Control de versiones (Git):
   - .gitignore: data/raw/*, *.pyc, .venv/, .ipynb_checkpoints/
   - Commits frecuentes con mensajes descriptivos

4. Ejecución:
   - Rutas relativas (pathlib) → portable
   - Para Colab: montar Drive y copiar archivos a data/raw/

5. Determinismo:
   - Sin operaciones aleatorias
   - Timestamps en nombres de archivos para trazabilidad



### 3.0 Reflexión final

**¿Cómo contribuye el flujo propuesto a la claridad del análisis?**

La separación en secciones (carga → validación → integración → análisis → exportación) permite leer el notebook como un documento lineal donde cada paso tiene un propósito claro. Las funciones reutilizables eliminan código repetido y hacen explícitas las reglas de validación que de otro modo quedarían implícitas en operaciones sueltas.

**¿Qué decisiones favorecen la eficiencia del proceso?**

Usar `validate='many_to_one'` en el merge detecta errores en el catálogo antes de que contaminen el dataset. Exportar en múltiples formatos con un loop evita código duplicado. Las rutas con pathlib y los timestamps en archivos eliminan la necesidad de renombrar manualmente.

**¿Qué elementos garantizan la trazabilidad?**

La columna 'campus' añadida antes del concat preserva el origen de cada fila. El indicador `_merge` identifica huérfanos. Los indicadores de calidad funcionan como un log de auditoría: si en la próxima corrida el número de huérfanos cambia de 5 a 20, sabemos que el catálogo se desactualizó.

**¿Qué riesgos aparecerían si este proceso se hiciera manualmente en una hoja de cálculo?**

Copiar y pegar entre hojas no deja registro de qué filas se unieron ni cuáles quedaron fuera. Un VLOOKUP roto falla silenciosamente devolviendo #N/A sin cuantificar el impacto. No hay versionamiento: si alguien sobrescribe el archivo, se pierde el estado anterior. Escalar a más campus o semestres implicaría repetir manualmente cada paso, multiplicando la probabilidad de error humano.